# Module 04 — ML Theory & Evaluation
## 04-10: Gaussian Processes & Bayesian Optimization

**Objective:** Build Gaussian processes from scratch — specifying kernel functions as
covariance, computing posterior distributions, and constructing Bayesian optimization
on top of GPs using Expected Improvement and UCB acquisition functions.

**Prerequisites:** 1-07 (Information Theory), 3-08 (Kernel Methods), 4-02 (Cross-Validation)


## Part 0 — Setup & Prerequisites

A **Gaussian Process** (GP) is a distribution over functions — instead of placing a prior
over model parameters, we place it directly in function space.

$$
f(\mathbf{x}) \sim \mathcal{GP}\bigl(\mu(\mathbf{x}),\; k(\mathbf{x}, \mathbf{x}')\bigr)
$$

The mean function $\mu$ (often zero) encodes prior beliefs about the output level.
The **kernel** $k(\mathbf{x}, \mathbf{x}')$ encodes the correlation structure —
it determines how quickly the function changes and what shapes are probable.

We build:
1. **Four kernel functions** — RBF, Matérn-5/2, Periodic, Linear.
2. **GP prior sampling** — draw functions from the prior.
3. **GP posterior** — condition on observations; obtain posterior mean + uncertainty.
4. **Bayesian optimization** — use a GP surrogate + acquisition function to find function optima.

**Prerequisites:** 1-07 (Information Theory), 3-08 (Kernel Methods), 4-02 (Cross-Validation)

> **Theory notebook:** every concept is implemented from scratch and empirically validated.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import math
import warnings
warnings.filterwarnings("ignore")
from typing import Any, Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.datasets import make_regression
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

from scipy.stats import norm as scipy_norm
from scipy.optimize import minimize as scipy_minimize

import torch

import random
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Seed: {SEED}")


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
GP_JITTER       = 1e-6    # numerical jitter added to kernel diagonal
N_PRIOR_SAMPLES = 5       # prior sample paths to draw
N_GP_GRID       = 300     # test grid resolution for GP prediction
N_BO_INIT       = 4       # initial random points in Bayesian optimisation
N_BO_ITER       = 12      # number of BO iterations
N_BO_CANDIDATES = 2000    # random candidates for acquisition maximisation
N_COVERAGE_RUNS = 500     # Monte Carlo runs for uncertainty coverage validation
NOISE_VAR       = 0.05    # observation noise variance for GP fits

# Forrester function domain
BO_LB, BO_UB   = 0.0, 1.0   # 1-D Bayesian optimisation bounds

print("Configuration:")
print(f"  GP_JITTER  = {GP_JITTER}  |  NOISE_VAR  = {NOISE_VAR}")
print(f"  N_BO_INIT  = {N_BO_INIT}   |  N_BO_ITER  = {N_BO_ITER}")
print(f"  BO domain  = [{BO_LB}, {BO_UB}]")


---
## Part 1 — Theoretical Foundation & Implementation

### 1.1 Kernel Functions as Covariance

The kernel (covariance) function $k(\mathbf{x}, \mathbf{x}')$ controls the GP's
inductive bias — it encodes assumptions about smoothness, periodicity, and scale.

| Kernel | Formula | Inductive Bias |
|--------|---------|----------------|
| **RBF** | $a^2 \exp\!\left(-\frac{\|\mathbf{x}-\mathbf{x}'\|^2}{2\ell^2}\right)$ | Infinitely differentiable, smooth |
| **Matérn-5/2** | $a^2(1+\frac{\sqrt{5}r}{\ell}+\frac{5r^2}{3\ell^2})e^{-\sqrt{5}r/\ell}$ | Twice differentiable, rougher |
| **Periodic** | $a^2\exp\!\left(-\frac{2\sin^2(\pi r/p)}{\ell^2}\right)$ | Periodic with period $p$ |
| **Linear** | $\sigma_b^2 + \sigma_v^2 (\mathbf{x}-c)(\mathbf{x}'-c)$ | Linear trends |

The kernel determines the **Gram matrix** $\mathbf{K}$ where $K_{ij} = k(\mathbf{x}_i, \mathbf{x}_j)$.


In [ ]:
# ── Part 1: Kernel Functions ───────────────────────────────────────────────────


def rbf_kernel(
    X1: np.ndarray,
    X2: np.ndarray,
    length_scale: float = 1.0,
    amplitude: float = 1.0,
) -> np.ndarray:
    '''Radial Basis Function (squared exponential) kernel.

    k(x, x') = amplitude^2 * exp(-||x - x'||^2 / (2 * length_scale^2))

    Args:
        X1: First input array, shape (n1, d) or (n1,) for 1-D.
        X2: Second input array, shape (n2, d) or (n2,) for 1-D.
        length_scale: Controls how quickly correlations decay with distance.
        amplitude: Overall output scale (signal standard deviation).

    Returns:
        Kernel matrix, shape (n1, n2).
    '''
    X1 = np.atleast_2d(X1)
    X2 = np.atleast_2d(X2)
    sq_dists = np.sum((X1[:, None, :] - X2[None, :, :]) ** 2, axis=-1)
    return float(amplitude) ** 2 * np.exp(-sq_dists / (2.0 * float(length_scale) ** 2))


def matern52_kernel(
    X1: np.ndarray,
    X2: np.ndarray,
    length_scale: float = 1.0,
    amplitude: float = 1.0,
) -> np.ndarray:
    '''Matérn kernel with smoothness parameter nu=5/2.

    k(x, x') = amplitude^2 * (1 + sqrt(5)*r/l + 5*r^2/(3*l^2)) * exp(-sqrt(5)*r/l)
    where r = ||x - x'||.

    Args:
        X1: First input array, shape (n1, d) or (n1,).
        X2: Second input array, shape (n2, d) or (n2,).
        length_scale: Length scale parameter.
        amplitude: Output scale.

    Returns:
        Kernel matrix, shape (n1, n2).
    '''
    X1 = np.atleast_2d(X1)
    X2 = np.atleast_2d(X2)
    sq_dists = np.sum((X1[:, None, :] - X2[None, :, :]) ** 2, axis=-1)
    r        = np.sqrt(np.maximum(sq_dists, 0.0))
    r_scaled = math.sqrt(5.0) * r / max(float(length_scale), 1e-9)
    return float(amplitude) ** 2 * (1.0 + r_scaled + r_scaled ** 2 / 3.0) * np.exp(-r_scaled)


def periodic_kernel(
    X1: np.ndarray,
    X2: np.ndarray,
    length_scale: float = 1.0,
    amplitude: float = 1.0,
    period: float = 1.0,
) -> np.ndarray:
    '''Periodic (Exp-Sine-Squared) kernel.

    k(x, x') = amplitude^2 * exp(-2 * sin^2(pi * |x - x'| / period) / length_scale^2)

    Args:
        X1: First input array, shape (n1, 1) or (n1,) for 1-D.
        X2: Second input array, shape (n2, 1) or (n2,) for 1-D.
        length_scale: Controls smoothness within each period.
        amplitude: Output scale.
        period: Period of the kernel.

    Returns:
        Kernel matrix, shape (n1, n2).
    '''
    X1 = np.atleast_2d(X1)
    X2 = np.atleast_2d(X2)
    dists = np.sqrt(np.sum((X1[:, None, :] - X2[None, :, :]) ** 2, axis=-1))
    sine_sq = np.sin(math.pi * dists / max(float(period), 1e-9)) ** 2
    return float(amplitude) ** 2 * np.exp(-2.0 * sine_sq / max(float(length_scale) ** 2, 1e-9))


def linear_kernel(
    X1: np.ndarray,
    X2: np.ndarray,
    variance: float = 1.0,
    offset: float = 0.0,
) -> np.ndarray:
    '''Linear (dot-product) kernel.

    k(x, x') = variance * (x - offset)^T (x' - offset)

    Args:
        X1: First input array, shape (n1, d) or (n1,).
        X2: Second input array, shape (n2, d) or (n2,).
        variance: Scale of the linear component.
        offset: Centre of the linear kernel.

    Returns:
        Kernel matrix, shape (n1, n2).
    '''
    X1 = np.atleast_2d(X1) - float(offset)
    X2 = np.atleast_2d(X2) - float(offset)
    return float(variance) * (X1 @ X2.T)


print("Kernels defined: rbf_kernel | matern52_kernel | periodic_kernel | linear_kernel")
# Quick sanity check: k(x, x) = amplitude^2 for RBF
x_check = np.array([[0.0], [1.0], [2.0]])
K_diag  = np.diag(rbf_kernel(x_check, x_check, length_scale=1.0, amplitude=2.0))
print(f"RBF diag (expect 4.0 each): {K_diag}")
K_diag_m = np.diag(matern52_kernel(x_check, x_check, length_scale=1.0, amplitude=1.5))
print(f"Matern52 diag (expect 2.25 each): {K_diag_m}")


In [ ]:
# ── Visualise Kernel Matrices and Sample Paths ─────────────────────────────────
rng_kv   = np.random.default_rng(SEED)
x_1d     = np.linspace(0, 5, 100).reshape(-1, 1)
n_paths  = N_PRIOR_SAMPLES

kernels_viz = [
    ("RBF (l=0.5)",   lambda X1, X2: rbf_kernel(X1, X2, 0.5, 1.0)),
    ("RBF (l=2.0)",   lambda X1, X2: rbf_kernel(X1, X2, 2.0, 1.0)),
    ("Matern-5/2",    lambda X1, X2: matern52_kernel(X1, X2, 1.0, 1.0)),
    ("Periodic(p=2)", lambda X1, X2: periodic_kernel(X1, X2, 0.5, 1.0, 2.0)),
]

fig_kv, axes_kv = plt.subplots(2, len(kernels_viz), figsize=(18, 7))

for col_k, (k_name, k_fn) in enumerate(kernels_viz):
    K_viz = k_fn(x_1d, x_1d)
    K_jit = K_viz + GP_JITTER * np.eye(len(x_1d))
    L_viz = np.linalg.cholesky(K_jit)
    samples_k = L_viz @ rng_kv.standard_normal((len(x_1d), n_paths))

    # Top row: kernel matrix
    im_k = axes_kv[0, col_k].imshow(K_viz, cmap="viridis", aspect="auto",
                                      vmin=0, vmax=1.05 * K_viz.max())
    axes_kv[0, col_k].set_title(k_name, fontsize=10, fontweight="bold")
    axes_kv[0, col_k].set_xticks([0, 50, 99])
    axes_kv[0, col_k].set_xticklabels(["0", "2.5", "5"])
    axes_kv[0, col_k].set_yticks([0, 50, 99])
    axes_kv[0, col_k].set_yticklabels(["0", "2.5", "5"])
    plt.colorbar(im_k, ax=axes_kv[0, col_k], fraction=0.046, pad=0.04)

    # Bottom row: prior sample paths
    for s in range(n_paths):
        axes_kv[1, col_k].plot(x_1d.ravel(), samples_k[:, s], alpha=0.8, lw=1.5)
    axes_kv[1, col_k].axhline(0, color="gray", ls=":", lw=1)
    axes_kv[1, col_k].set_xlabel("x")
    axes_kv[1, col_k].set_ylabel("f(x)" if col_k == 0 else "")
    axes_kv[1, col_k].set_title(f"{k_name} — prior paths", fontsize=9)

axes_kv[0, 0].set_ylabel("Kernel matrix $k(x, x')$")
plt.suptitle("Kernel Matrices (top) and GP Prior Sample Paths (bottom)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print("RBF small l: wiggly samples.  RBF large l: smooth samples.")
print("Matern-5/2: slightly rougher than RBF.  Periodic: repeating patterns.")


### 1.2 GP Prior — Drawing Functions from a Gaussian Prior

Before observing any data, the GP prior defines a joint distribution over function values
at any finite set of input points:
$$
\mathbf{f} = \bigl[f(\mathbf{x}_1), \ldots, f(\mathbf{x}_n)\bigr]^\top
\sim \mathcal{N}(\mathbf{0},\; \mathbf{K})
$$
where $K_{ij} = k(\mathbf{x}_i, \mathbf{x}_j)$.  Sampling is done via Cholesky
decomposition: $\mathbf{f} = \mathbf{L}\boldsymbol{\varepsilon}$, $\boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$.


In [ ]:
# ── GP Prior Sampling ──────────────────────────────────────────────────────────


def gp_prior_samples(
    X: np.ndarray,
    kernel_fn: Callable[[np.ndarray, np.ndarray], np.ndarray],
    n_samples: int = 5,
    jitter: float = GP_JITTER,
) -> np.ndarray:
    '''Draw samples from a zero-mean GP prior.

    Computes the Gram matrix K and returns samples via K = L L^T decomposition.

    Args:
        X: Input array, shape (n, d) or (n, 1) for 1-D.
        kernel_fn: Callable (X1, X2) -> (n1, n2) kernel matrix.
        n_samples: Number of sample paths to draw.
        jitter: Small diagonal added for numerical stability.

    Returns:
        Sample paths, shape (n, n_samples).
    '''
    K  = kernel_fn(X, X) + jitter * np.eye(len(X))
    L  = np.linalg.cholesky(K)
    eps = np.random.randn(len(X), n_samples)
    return L @ eps


# Demo: GP prior on 1-D domain with different kernels
x_demo = np.linspace(-3, 3, 200).reshape(-1, 1)

fig_pr, axes_pr = plt.subplots(1, 3, figsize=(15, 4))
k_demo_configs = [
    ("RBF l=0.3 (wiggly)", lambda X1, X2: rbf_kernel(X1, X2, 0.3, 1.0)),
    ("RBF l=1.5 (smooth)",  lambda X1, X2: rbf_kernel(X1, X2, 1.5, 1.0)),
    ("Matern-5/2 l=0.8",   lambda X1, X2: matern52_kernel(X1, X2, 0.8, 1.0)),
]
for ax_pr, (k_label_pr, k_fn_pr) in zip(axes_pr, k_demo_configs):
    paths_pr = gp_prior_samples(x_demo, k_fn_pr, n_samples=6)
    for path_p in paths_pr.T:
        ax_pr.plot(x_demo.ravel(), path_p, alpha=0.75, lw=1.5)
    ax_pr.axhline(0, color="gray", ls=":", lw=1)
    ax_pr.fill_between(x_demo.ravel(),
                        -2 * np.ones(len(x_demo)),
                         2 * np.ones(len(x_demo)),
                        alpha=0.08, color="gray", label="95% prior interval")
    ax_pr.set_title(k_label_pr, fontsize=10, fontweight="bold")
    ax_pr.set_xlabel("x"); ax_pr.set_ylim(-3.5, 3.5)
    ax_pr.legend(fontsize=8)
axes_pr[0].set_ylabel("f(x)")
plt.suptitle("GP Prior Sample Paths: Effect of Kernel Choice", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print("The kernel completely determines the function space the GP explores.")


### 1.3 GP Posterior — Conditioning on Observations

Given $n$ noisy observations $y_i = f(\mathbf{x}_i) + \varepsilon_i$,
$\varepsilon_i \sim \mathcal{N}(0, \sigma_n^2)$, the GP posterior at test points $\mathbf{X}^*$ is:

$$
p(\mathbf{f}^* \mid \mathbf{X}^*, \mathbf{X}, \mathbf{y}) = \mathcal{N}(\boldsymbol{\mu}^*, \boldsymbol{\Sigma}^*)
$$

$$
\boldsymbol{\mu}^* = \mathbf{K}_{*} (\mathbf{K} + \sigma_n^2 \mathbf{I})^{-1} \mathbf{y},
\quad
\boldsymbol{\Sigma}^* = \mathbf{K}_{**} - \mathbf{K}_{*}(\mathbf{K} + \sigma_n^2\mathbf{I})^{-1}\mathbf{K}_{*}^\top
$$

Efficient computation via **Cholesky decomposition** $\mathbf{L}\mathbf{L}^\top = \mathbf{K} + \sigma_n^2\mathbf{I}$:

$$
\boldsymbol{\alpha} = (\mathbf{K} + \sigma_n^2\mathbf{I})^{-1}\mathbf{y},
\quad \boldsymbol{\mu}^* = \mathbf{K}_* \boldsymbol{\alpha},
\quad \boldsymbol{\Sigma}^* = \mathbf{K}_{**} - \mathbf{v}^\top\mathbf{v}
\quad \text{where } \mathbf{v} = \mathbf{L}^{-1}\mathbf{K}_*^\top
$$

**Log marginal likelihood** (used for kernel hyperparameter tuning):
$$
\log p(\mathbf{y} \mid \mathbf{X}) = -\tfrac{1}{2}\mathbf{y}^\top\boldsymbol{\alpha}
  - \sum_i \log L_{ii} - \tfrac{n}{2}\log 2\pi
$$


In [ ]:
# ── Part 1: GaussianProcess Class ─────────────────────────────────────────────


class GaussianProcess:
    '''Exact Gaussian Process regression with Cholesky-based inference.

    Implements GP posterior prediction, posterior sampling, and log marginal
    likelihood computation for kernel hyperparameter selection.

    Attributes:
        kernel_fn: Callable (X1, X2) -> kernel matrix.
        noise_var: Observation noise variance sigma_n^2.
        jitter: Small diagonal term for numerical stability.
        X_train: Training inputs after fitting, shape (n, d).
        y_train: Training targets after fitting, shape (n,).
    '''

    def __init__(
        self,
        kernel_fn: Callable[[np.ndarray, np.ndarray], np.ndarray],
        noise_var: float = 1e-3,
        jitter: float = GP_JITTER,
    ) -> None:
        '''Initialise GaussianProcess with kernel, noise variance, and jitter.

        Args:
            kernel_fn: Callable (X1, X2) -> (n1, n2) covariance matrix.
            noise_var: Observation noise variance.
            jitter: Small diagonal noise for Cholesky stability.
        '''
        self.kernel_fn: Callable = kernel_fn
        self.noise_var: float    = float(noise_var)
        self.jitter:    float    = float(jitter)
        self.X_train:   np.ndarray | None = None
        self.y_train:   np.ndarray | None = None
        self._L:     np.ndarray | None = None
        self._alpha: np.ndarray | None = None

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
    ) -> "GaussianProcess":
        '''Store training data and pre-compute the Cholesky factorisation.

        Args:
            X_train: Training inputs, shape (n, d) or (n, 1) for 1-D.
            y_train: Training targets (zero-mean recommended), shape (n,).

        Returns:
            Self (fitted GP).
        '''
        self.X_train = np.atleast_2d(X_train)
        self.y_train = np.asarray(y_train, dtype=float)
        n    = len(y_train)
        K    = self.kernel_fn(self.X_train, self.X_train)
        K   += (self.noise_var + self.jitter) * np.eye(n)
        self._L     = np.linalg.cholesky(K)
        self._alpha = np.linalg.solve(
            self._L.T, np.linalg.solve(self._L, self.y_train)
        )
        return self

    def predict(
        self,
        X_test: np.ndarray,
    ) -> tuple[np.ndarray, np.ndarray]:
        '''Compute posterior mean and predictive standard deviation.

        Args:
            X_test: Test inputs, shape (n_test, d) or (n_test, 1).

        Returns:
            Tuple (mu_post, sigma_post) each shape (n_test,).
        '''
        X_test   = np.atleast_2d(X_test)
        K_star   = self.kernel_fn(X_test, self.X_train)
        mu_post  = K_star @ self._alpha
        v        = np.linalg.solve(self._L, K_star.T)
        K_ss_diag = np.diag(self.kernel_fn(X_test, X_test))
        var_post  = K_ss_diag - np.sum(v ** 2, axis=0)
        var_post  = np.maximum(var_post, 0.0)
        return mu_post, np.sqrt(var_post)

    def sample_posterior(
        self,
        X_test: np.ndarray,
        n_samples: int = 5,
    ) -> np.ndarray:
        '''Draw sample paths from the GP posterior.

        Args:
            X_test: Test inputs, shape (n_test, d).
            n_samples: Number of posterior sample paths.

        Returns:
            Sample paths, shape (n_test, n_samples).
        '''
        X_test  = np.atleast_2d(X_test)
        K_star  = self.kernel_fn(X_test, self.X_train)
        mu_post = K_star @ self._alpha
        v       = np.linalg.solve(self._L, K_star.T)
        K_ss    = self.kernel_fn(X_test, X_test)
        cov_post = K_ss - v.T @ v
        cov_post += self.jitter * np.eye(len(X_test))
        L_post   = np.linalg.cholesky(cov_post)
        return mu_post[:, None] + L_post @ np.random.randn(len(X_test), n_samples)

    def log_marginal_likelihood(self) -> float:
        '''Compute log p(y | X) for kernel hyperparameter selection.

        log p(y|X) = -0.5 * y^T alpha - sum(log diag(L)) - n/2 * log(2*pi)

        Returns:
            Log marginal likelihood as float.
        '''
        n   = len(self.y_train)
        lml = (-0.5 * float(self.y_train @ self._alpha)
               - float(np.sum(np.log(np.diag(self._L))))
               - 0.5 * n * math.log(2.0 * math.pi))
        return lml


print("GaussianProcess class defined.")
print("  Methods: fit() | predict() | sample_posterior() | log_marginal_likelihood()")


In [ ]:
# ── Fit GP to 1-D Observations and Visualise Posterior ────────────────────────
rng_gp   = np.random.default_rng(SEED + 10)

# True latent function and observations
true_fn   = lambda x: np.sin(2.0 * np.pi * x / 3.0) + 0.5 * x
X_obs_1d  = rng_gp.uniform(-3, 3, size=8).reshape(-1, 1)
y_obs_1d  = true_fn(X_obs_1d.ravel()) + rng_gp.normal(0, 0.3, size=8)

x_test_1d = np.linspace(-4, 4, N_GP_GRID).reshape(-1, 1)
y_true_1d = true_fn(x_test_1d.ravel())

# Fit GP with RBF kernel
k_rbf_fit = lambda X1, X2: rbf_kernel(X1, X2, length_scale=1.2, amplitude=1.5)
gp_fit    = GaussianProcess(k_rbf_fit, noise_var=NOISE_VAR)
gp_fit.fit(X_obs_1d, y_obs_1d)

mu_fit, sigma_fit  = gp_fit.predict(x_test_1d)
post_samples_fit   = gp_fit.sample_posterior(x_test_1d, n_samples=4)

fig_gp, axes_gp = plt.subplots(1, 2, figsize=(14, 5))

# Left: posterior mean + uncertainty bands
ax_left = axes_gp[0]
ax_left.plot(x_test_1d.ravel(), y_true_1d, "k--", lw=1.5, label="True f(x)", zorder=5)
ax_left.plot(x_test_1d.ravel(), mu_fit, "steelblue", lw=2, label="GP posterior mean")
ax_left.fill_between(x_test_1d.ravel(),
                      mu_fit - 2 * sigma_fit,
                      mu_fit + 2 * sigma_fit,
                      alpha=0.25, color="steelblue", label="2$\sigma$ band (95%)")
ax_left.fill_between(x_test_1d.ravel(),
                      mu_fit - sigma_fit,
                      mu_fit + sigma_fit,
                      alpha=0.35, color="steelblue", label="1$\sigma$ band (68%)")
ax_left.scatter(X_obs_1d.ravel(), y_obs_1d, c="tomato", s=60, zorder=10,
                label="Observations", edgecolors="k", lw=0.5)
ax_left.set_title("GP Posterior: Mean and Uncertainty Bands", fontsize=11, fontweight="bold")
ax_left.set_xlabel("x"); ax_left.set_ylabel("f(x)")
ax_left.legend(fontsize=8); ax_left.set_xlim(-4, 4)

# Right: posterior sample paths
ax_right = axes_gp[1]
ax_right.plot(x_test_1d.ravel(), y_true_1d, "k--", lw=1.5, label="True f(x)", zorder=5)
for s in range(post_samples_fit.shape[1]):
    ax_right.plot(x_test_1d.ravel(), post_samples_fit[:, s],
                   alpha=0.7, lw=1.2)
ax_right.scatter(X_obs_1d.ravel(), y_obs_1d, c="tomato", s=60, zorder=10,
                  edgecolors="k", lw=0.5, label="Observations")
ax_right.set_title("GP Posterior: Sample Paths (conditioned on data)",
                    fontsize=11, fontweight="bold")
ax_right.set_xlabel("x"); ax_right.legend(fontsize=8)
ax_right.set_xlim(-4, 4)

plt.suptitle("GP Regression: RBF Kernel (l=1.2, a=1.5)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

lml_fit = gp_fit.log_marginal_likelihood()
print(f"GP posterior fit:")
print(f"  Log marginal likelihood: {lml_fit:.4f}")
print(f"  Mean sigma at obs points: {float(gp_fit.predict(X_obs_1d)[1].mean()):.5f}  (near 0 = interpolates data)")
print(f"  Mean sigma far from data: {float(gp_fit.predict(np.array([[5.0]]) )[1]):.5f}  (large = uncertain far from data)")


### 1.4 Kernel Hyperparameter Tuning via Log Marginal Likelihood

The kernel hyperparameters ($\ell$, $a$, $\sigma_n^2$) are found by maximising the
log marginal likelihood $\log p(\mathbf{y} \mid \mathbf{X})$.

This is the **type-II maximum likelihood** approach — it balances data fit (likelihood)
against model complexity (Occam's razor encoded in the prior normalization).


In [ ]:
# ── LML Surface Over (length_scale, amplitude) Grid ───────────────────────────
ls_grid_vals  = np.logspace(-1.5, 1.0, 30)   # length_scale
amp_grid_vals = np.logspace(-0.5, 1.0, 30)   # amplitude
lml_surface   = np.zeros((len(ls_grid_vals), len(amp_grid_vals)))

for i, ls_v in enumerate(ls_grid_vals):
    for j, amp_v in enumerate(amp_grid_vals):
        k_ij = lambda X1, X2, l=ls_v, a=amp_v: rbf_kernel(X1, X2, l, a)
        gp_ij = GaussianProcess(k_ij, noise_var=NOISE_VAR)
        try:
            gp_ij.fit(X_obs_1d, y_obs_1d)
            lml_surface[i, j] = gp_ij.log_marginal_likelihood()
        except np.linalg.LinAlgError:
            lml_surface[i, j] = float("-inf")

# Find the maximum
best_ij     = np.unravel_index(np.argmax(lml_surface), lml_surface.shape)
best_ls     = float(ls_grid_vals[best_ij[0]])
best_amp    = float(amp_grid_vals[best_ij[1]])
best_lml    = float(lml_surface[best_ij])

fig_lml, ax_lml = plt.subplots(figsize=(8, 6))
im_lml = ax_lml.contourf(
    np.log10(amp_grid_vals), np.log10(ls_grid_vals),
    lml_surface, levels=30, cmap="viridis",
)
ax_lml.contour(
    np.log10(amp_grid_vals), np.log10(ls_grid_vals),
    lml_surface, levels=10, colors="white", linewidths=0.5, alpha=0.4,
)
ax_lml.scatter(np.log10(best_amp), np.log10(best_ls),
               c="tomato", s=150, zorder=5, label=f"Best LML: ls={best_ls:.3f}, amp={best_amp:.3f}")
ax_lml.set_xlabel("log10(amplitude)", fontsize=11)
ax_lml.set_ylabel("log10(length_scale)", fontsize=11)
ax_lml.set_title("Log Marginal Likelihood vs RBF Hyperparameters",
                  fontsize=11, fontweight="bold")
ax_lml.legend(fontsize=9)
plt.colorbar(im_lml, ax=ax_lml, label="LML")
plt.tight_layout()
plt.show()

# Fit GP with optimal hyperparameters
k_opt    = lambda X1, X2: rbf_kernel(X1, X2, best_ls, best_amp)
gp_opt   = GaussianProcess(k_opt, noise_var=NOISE_VAR)
gp_opt.fit(X_obs_1d, y_obs_1d)
mu_opt, sigma_opt = gp_opt.predict(x_test_1d)
print(f"Optimal hyperparameters from LML grid search:")
print(f"  length_scale = {best_ls:.4f}  amplitude = {best_amp:.4f}")
print(f"  LML = {best_lml:.4f}  (initial LML = {lml_fit:.4f})")
print(f"  LML increase: {best_lml - lml_fit:.4f}")


### 1.5 Optimising GP Hyperparameters via Gradient Ascent on LML

Grid search over hyperparameters is expensive ($O(G^2)$ GP fits). A better approach is to **maximise the LML directly** using a gradient-based optimiser (L-BFGS-B) in log-space.  We run 5 restarts to avoid local optima and compare the optimised GP against the default hyperparameters.


In [ ]:
# ── LML Hyperparameter Optimisation via scipy.optimize ───────────────────────
# Instead of grid search, we maximise the log marginal likelihood (LML) using
# L-BFGS-B.  This is exact GP hyperparameter learning (no approximation).
# We optimise log(length_scale) and log(amplitude) to avoid positivity
# constraints; the optimizer works in unconstrained log-space.


def negative_lml_rbf(
    log_params: np.ndarray,
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    noise_var: float = NOISE_VAR,
) -> float:
    '''Return negative LML for RBF kernel at given log-parameters.

    Args:
        log_params: Array [log_length_scale, log_amplitude].
        X_tr: Training inputs of shape (n,) or (n, 1).
        y_tr: Training targets of shape (n,).
        noise_var: Fixed observation noise variance.

    Returns:
        Negative log marginal likelihood (scalar).
    '''
    ls_opt  = float(np.exp(log_params[0]))
    amp_opt = float(np.exp(log_params[1]))
    kernel_opt = lambda X1, X2: rbf_kernel(X1, X2, ls_opt, amp_opt)
    gp_tmp = GaussianProcess(kernel_opt, noise_var=noise_var)
    gp_tmp.fit(X_tr, y_tr)
    return -gp_tmp.log_marginal_likelihood()


# Optimise from 5 random starting points and keep the best result
rng_opt = np.random.default_rng(SEED + 20)
X_opt_tr = gp_1d_X  # reuse 1-D demo data from cell above
y_opt_tr = gp_1d_y

best_nlml   = float("inf")
best_ls_opt = 1.0
best_amp_opt = 1.0
n_restarts  = 5

print("LML Hyperparameter Optimisation — RBF Kernel (5 restarts)")
print(f"{'Restart':>7s}  {'Init ls':>8s}  {'Init amp':>9s}  {'Final ls':>9s}  {'Final amp':>10s}  {'LML':>10s}")
print("-" * 62)

for restart in range(n_restarts):
    log_ls0  = rng_opt.uniform(-1.5, 1.5)
    log_amp0 = rng_opt.uniform(-1.0, 1.5)
    x0 = np.array([log_ls0, log_amp0])
    res = scipy_minimize(
        negative_lml_rbf,
        x0,
        args=(X_opt_tr[:, None], y_opt_tr),
        method="L-BFGS-B",
        bounds=[(-3.0, 3.0), (-2.0, 3.0)],
    )
    ls_r   = float(np.exp(res.x[0]))
    amp_r  = float(np.exp(res.x[1]))
    lml_r  = float(-res.fun)
    print(f"{restart + 1:7d}  {np.exp(log_ls0):8.4f}  {np.exp(log_amp0):9.4f}  "
          f"{ls_r:9.4f}  {amp_r:10.4f}  {lml_r:10.4f}")
    if res.fun < best_nlml:
        best_nlml    = res.fun
        best_ls_opt  = ls_r
        best_amp_opt = amp_r

print(f"\nOptimised  ->  length_scale={best_ls_opt:.4f}, amplitude={best_amp_opt:.4f}, "
      f"LML={-best_nlml:.4f}")

# Compare optimised GP vs default-hyperparameter GP on test grid
X_test_lml = np.linspace(-5, 5, 300)[:, None]
kernel_default = lambda X1, X2: rbf_kernel(X1, X2, 1.0, 1.0)
kernel_optim   = lambda X1, X2: rbf_kernel(X1, X2, best_ls_opt, best_amp_opt)

gp_default = GaussianProcess(kernel_default, noise_var=NOISE_VAR)
gp_default.fit(X_opt_tr[:, None], y_opt_tr)
mu_def, std_def = gp_default.predict(X_test_lml)

gp_optim = GaussianProcess(kernel_optim, noise_var=NOISE_VAR)
gp_optim.fit(X_opt_tr[:, None], y_opt_tr)
mu_opt, std_opt = gp_optim.predict(X_test_lml)

lml_default = gp_default.log_marginal_likelihood()
lml_optimised = gp_optim.log_marginal_likelihood()

fig_lml, axes_lml = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
x_test_1d = X_test_lml.ravel()
obs_x = X_opt_tr.ravel()

for ax, mu, std, title, lml_val in zip(
    axes_lml,
    [mu_def, mu_opt],
    [std_def, std_opt],
    [f"Default (ls=1, amp=1)", f"Optimised (ls={best_ls_opt:.2f}, amp={best_amp_opt:.2f})"],
    [lml_default, lml_optimised],
):
    ax.fill_between(x_test_1d, mu - 2 * std, mu + 2 * std, alpha=0.25, label="±2σ")
    ax.plot(x_test_1d, mu, lw=2, label="Posterior mean")
    ax.scatter(obs_x, y_opt_tr, c="red", zorder=5, s=40, label="Observations")
    ax.set_title(f"{title}\nLML = {lml_val:.2f}", fontsize=10, fontweight="bold")
    ax.set_xlabel("x")
    ax.set_ylabel("f(x)")
    ax.legend(fontsize=8)

plt.suptitle("LML Hyperparameter Optimisation: Default vs Optimised GP",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"LML improvement: {lml_optimised - lml_default:+.3f} nats")
print("Optimised GP fits the data more precisely (tighter posterior).")


---
## Part 2 — Empirical Validation

We validate two GP properties:
1. **Uncertainty coverage:** the $2\sigma$ predictive interval should contain ~95% of
   held-out observations for a well-calibrated GP.
2. **Kernel comparison:** different kernels express different inductive biases — we show
   how the same data produces very different posteriors with different kernels.


In [ ]:
# ── GP Uncertainty Band Coverage Validation ────────────────────────────────────
# Theory: if the GP model is correct, 2-sigma bands contain 95% of observations.
# We generate many synthetic datasets and compute empirical coverage.

rng_cov  = np.random.default_rng(SEED + 100)
sigma_multipliers = [1.0, 1.5, 2.0, 2.576]  # 68%, 86.6%, 95%, 99%
theoretical_cov   = [0.683, 0.866, 0.954, 0.990]

coverage_results: list = []
n_cov_obs  = 10     # training points per run
n_cov_test = 50     # test points per run

for mult_v, theory_v in zip(sigma_multipliers, theoretical_cov):
    covered_count = 0
    total_count   = 0
    for _ in range(N_COVERAGE_RUNS):
        X_cov_tr  = rng_cov.uniform(-3, 3, n_cov_obs).reshape(-1, 1)
        y_cov_tr  = true_fn(X_cov_tr.ravel()) + rng_cov.normal(0, 0.3, n_cov_obs)
        X_cov_te  = rng_cov.uniform(-3, 3, n_cov_test).reshape(-1, 1)
        y_cov_te  = true_fn(X_cov_te.ravel()) + rng_cov.normal(0, 0.3, n_cov_test)

        k_cov     = lambda X1, X2: rbf_kernel(X1, X2, 1.2, 1.5)
        gp_cov    = GaussianProcess(k_cov, noise_var=NOISE_VAR)
        gp_cov.fit(X_cov_tr, y_cov_tr)
        mu_c, sig_c = gp_cov.predict(X_cov_te)

        covered   = np.abs(y_cov_te - mu_c) <= mult_v * sig_c
        covered_count += int(covered.sum())
        total_count   += n_cov_test

    emp_cov = covered_count / max(total_count, 1)
    coverage_results.append((mult_v, theory_v, emp_cov))

print(f"GP uncertainty band coverage ({N_COVERAGE_RUNS} MC runs, {n_cov_test} test pts each):")
hdr_sig = "Sigma mult"; hdr_theory = "Theory"; hdr_emp = "Empirical"; hdr_ok = "Close?"
print(f"  {hdr_sig:>10s}  {hdr_theory:>8s}  {hdr_emp:>10s}  {hdr_ok:>6s}")
print("  " + "-" * 42)
for mult_v, theory_v, emp_v in coverage_results:
    close = "YES" if abs(emp_v - theory_v) < 0.05 else "NO"
    print(f"  {mult_v:>10.1f}  {theory_v:>8.3f}  {emp_v:>10.4f}  {close:>6s}")
print("")
print("GP coverage matches theory when the model and noise assumptions are correct.")
print("Miscalibration occurs if true noise differs from assumed noise_var.")


### 2.2 Kernel Comparison on the Same Data

The kernel encodes our prior beliefs about the function.  Applying different kernels
to the same observations produces different posteriors — demonstrating that choosing
the right kernel (or kernel mixture) is a critical modelling decision.


In [ ]:
# ── Compare Four Kernels on the Same Observation Set ──────────────────────────
kernels_cmp = [
    ("RBF l=0.5",    lambda X1, X2: rbf_kernel(X1, X2, 0.5, 1.0)),
    ("RBF l=2.0",    lambda X1, X2: rbf_kernel(X1, X2, 2.0, 1.0)),
    ("Matern-5/2",   lambda X1, X2: matern52_kernel(X1, X2, 0.8, 1.0)),
    ("Periodic p=3", lambda X1, X2: periodic_kernel(X1, X2, 0.8, 1.0, 3.0)),
]

fig_cmp, axes_cmp = plt.subplots(2, 2, figsize=(14, 9))
ax_flat_cmp = axes_cmp.ravel()
colours_cmp = ["steelblue", "darkorange", "green", "purple"]

for ax_c, (k_name_c, k_fn_c), col_c in zip(ax_flat_cmp, kernels_cmp, colours_cmp):
    gp_c = GaussianProcess(k_fn_c, noise_var=NOISE_VAR)
    gp_c.fit(X_obs_1d, y_obs_1d)
    mu_c, sig_c = gp_c.predict(x_test_1d)
    lml_c       = gp_c.log_marginal_likelihood()

    ax_c.plot(x_test_1d.ravel(), y_true_1d, "k--", lw=1.5, label="True f(x)")
    ax_c.plot(x_test_1d.ravel(), mu_c, color=col_c, lw=2, label="GP mean")
    ax_c.fill_between(x_test_1d.ravel(), mu_c - 2*sig_c, mu_c + 2*sig_c,
                       alpha=0.25, color=col_c, label="2$\sigma$")
    ax_c.scatter(X_obs_1d.ravel(), y_obs_1d, c="tomato", s=50, zorder=10,
                  edgecolors="k", lw=0.5)
    lml_tag = f" | LML={lml_c:.2f}"
    ax_c.set_title(k_name_c + lml_tag, fontsize=10, fontweight="bold")
    ax_c.set_xlabel("x"); ax_c.legend(fontsize=7)
    ax_c.set_xlim(-4, 4)

plt.suptitle("Kernel Comparison: Same Data, Different Priors",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Print LML for each kernel
print("Log marginal likelihood comparison:")
for k_name_c, k_fn_c in kernels_cmp:
    gp_c = GaussianProcess(k_fn_c, noise_var=NOISE_VAR)
    gp_c.fit(X_obs_1d, y_obs_1d)
    print(f"  {k_name_c:>18s}: LML = {gp_c.log_marginal_likelihood():.4f}")
print("")
print("Higher LML = better fit to the data under the model prior.")
print("Periodic kernel gets low LML on this non-periodic function.")


---
## Part 3 — Application: Bayesian Optimization

### 3.1 Bayesian Optimization Framework

**Bayesian optimization (BO)** efficiently searches for the minimum of an
**expensive black-box function** $f: \mathcal{X} \to \mathbb{R}$:

1. **Surrogate model:** a GP $p(f \mid \mathcal{D})$ fitted to all current observations.
2. **Acquisition function:** $\alpha(x)$ encodes the trade-off between exploration
   (uncertain regions) and exploitation (promising regions).
3. **Next evaluation:** $x_{n+1} = \arg\max_x \alpha(x)$.
4. Evaluate $y_{n+1} = f(x_{n+1})$, add to $\mathcal{D}$, and repeat.

**Test function — Forrester (1D benchmark):**
$$
f(x) = (6x - 2)^2 \sin(12x - 4), \quad x \in [0, 1]
$$
Global minimum at $x^* \approx 0.757$, $f^* \approx -6.02$.


### 3.2 Acquisition Functions

| Acquisition | Formula | Trade-off |
|-------------|---------|-----------|
| **Expected Improvement (EI)** | $\mathbb{E}[\max(f^* - f(x) - \xi, 0)]$ | Explores & exploits; $\xi$ controls exploration |
| **Upper Confidence Bound (UCB)** | $\mu(x) - \kappa\,\sigma(x)$ | $\kappa$ directly controls exploration vs exploitation |
| **Probability of Improvement (PI)** | $P(f(x) < f^* - \xi)$ | Greedier than EI; prone to local optima |

For **EI** with analytic Gaussian form:
$$
\text{EI}(x) = (f^* - \mu(x) - \xi)\,\Phi(z) + \sigma(x)\,\phi(z),
\quad z = \frac{f^* - \mu(x) - \xi}{\sigma(x)}
$$
where $\Phi$ and $\phi$ are the standard normal CDF and PDF.


In [ ]:
# ── Part 3: Acquisition Functions ─────────────────────────────────────────────


def expected_improvement(
    X_cands: np.ndarray,
    gp: GaussianProcess,
    f_best: float,
    xi: float = 0.01,
) -> np.ndarray:
    '''Compute Expected Improvement acquisition for minimisation.

    EI(x) = (f_best - mu(x) - xi) * Phi(z) + sigma(x) * phi(z)
    where z = (f_best - mu(x) - xi) / sigma(x).

    Args:
        X_cands: Candidate points, shape (n_cands, d).
        gp: Fitted GaussianProcess instance.
        f_best: Current best observed function value.
        xi: Exploration parameter (larger = more exploration).

    Returns:
        EI values, shape (n_cands,).
    '''
    mu_c, sig_c = gp.predict(X_cands)
    imp   = f_best - mu_c - xi
    z     = imp / (sig_c + 1e-9)
    ei    = imp * scipy_norm.cdf(z) + sig_c * scipy_norm.pdf(z)
    return np.maximum(ei, 0.0)


def upper_confidence_bound(
    X_cands: np.ndarray,
    gp: GaussianProcess,
    kappa: float = 2.0,
) -> np.ndarray:
    '''Compute Upper Confidence Bound acquisition for minimisation.

    UCB_min(x) = mu(x) - kappa * sigma(x)  (we maximise the negative for min.)

    Args:
        X_cands: Candidate points, shape (n_cands, d).
        gp: Fitted GaussianProcess instance.
        kappa: Exploration-exploitation trade-off.  Higher = more exploration.

    Returns:
        Negative UCB values (negate to turn max -> min), shape (n_cands,).
    '''
    mu_c, sig_c = gp.predict(X_cands)
    return -(mu_c - kappa * sig_c)


def probability_of_improvement(
    X_cands: np.ndarray,
    gp: GaussianProcess,
    f_best: float,
    xi: float = 0.01,
) -> np.ndarray:
    '''Compute Probability of Improvement acquisition for minimisation.

    PI(x) = Phi((f_best - mu(x) - xi) / sigma(x))

    Args:
        X_cands: Candidate points, shape (n_cands, d).
        gp: Fitted GaussianProcess instance.
        f_best: Current best observed function value.
        xi: Small improvement threshold.

    Returns:
        PI values in [0, 1], shape (n_cands,).
    '''
    mu_c, sig_c = gp.predict(X_cands)
    z = (f_best - mu_c - xi) / (sig_c + 1e-9)
    return scipy_norm.cdf(z)


# ── Test function: Forrester (1D BO benchmark) ────────────────────────────────
def forrester(x: np.ndarray) -> np.ndarray:
    '''Forrester 1D test function for Bayesian optimisation benchmarks.

    f(x) = (6x - 2)^2 * sin(12x - 4)
    Global minimum at x* ~ 0.757, f* ~ -6.02.

    Args:
        x: Input values in [0, 1], shape (n,) or scalar.

    Returns:
        Function values, same shape as x.
    '''
    x_f = np.asarray(x, dtype=float)
    return (6.0 * x_f - 2.0) ** 2 * np.sin(12.0 * x_f - 4.0)


x_forr = np.linspace(BO_LB, BO_UB, 500)
y_forr = forrester(x_forr)
x_opt_true  = float(x_forr[np.argmin(y_forr)])
f_opt_true  = float(y_forr.min())
print(f"Forrester function: global min at x*={x_opt_true:.4f}, f*={f_opt_true:.4f}")
print(f"Acquisition functions defined: expected_improvement | upper_confidence_bound | probability_of_improvement")


In [ ]:
# ── Part 3: BayesianOptimizer Class ───────────────────────────────────────────


class BayesianOptimizer:
    '''Bayesian optimisation loop using a GP surrogate model.

    Iteratively suggests evaluation points by maximising an acquisition function
    over random candidates within the specified bounds.

    Attributes:
        bounds: Tuple (lower, upper) of scalar bounds for 1-D optimisation.
        acquisition: Acquisition type: 'ei', 'ucb', or 'pi'.
        kappa: UCB exploration parameter.
        xi: EI / PI exploration parameter.
        X_observed: All observed inputs, shape (n, 1).
        y_observed: All observed outputs, shape (n,).
        best_x: Best x found so far.
        best_y: Best function value found so far.
    '''

    def __init__(
        self,
        bounds: tuple[float, float],
        acquisition: str = "ei",
        kappa: float = 2.0,
        xi: float = 0.01,
    ) -> None:
        '''Initialise BayesianOptimizer with bounds and acquisition settings.

        Args:
            bounds: (lower_bound, upper_bound) for 1-D domain.
            acquisition: Acquisition function: "ei", "ucb", or "pi".
            kappa: Exploration parameter for UCB.
            xi: Exploration parameter for EI and PI.
        '''
        self.bounds:      tuple[float, float] = bounds
        self.acquisition: str   = acquisition
        self.kappa:       float = kappa
        self.xi:          float = xi
        self.X_observed:  np.ndarray | None = None
        self.y_observed:  np.ndarray | None = None
        self.best_x:      float = float("nan")
        self.best_y:      float = float("inf")

    def observe(
        self,
        x_new: float | np.ndarray,
        y_new: float,
    ) -> None:
        '''Record a new (x, y) observation.

        Args:
            x_new: New input value (scalar or 1-D array).
            y_new: Observed function value.
        '''
        x_arr = np.atleast_2d(float(x_new) if np.isscalar(x_new) else x_new)
        if self.X_observed is None:
            self.X_observed = x_arr
            self.y_observed = np.array([float(y_new)])
        else:
            self.X_observed = np.vstack([self.X_observed, x_arr])
            self.y_observed = np.append(self.y_observed, float(y_new))
        if float(y_new) < self.best_y:
            self.best_y = float(y_new)
            self.best_x = float(x_new if np.isscalar(x_new) else x_new.ravel()[0])

    def suggest(
        self,
        kernel_fn: Callable[[np.ndarray, np.ndarray], np.ndarray],
        n_candidates: int = N_BO_CANDIDATES,
    ) -> float:
        '''Suggest the next evaluation point by maximising the acquisition function.

        Fits a GP on all current observations, evaluates the acquisition over
        random candidates, and returns the candidate with the highest value.

        Args:
            kernel_fn: Kernel function for the GP surrogate.
            n_candidates: Number of random candidates to evaluate.

        Returns:
            Suggested next x value as float.
        '''
        gp = GaussianProcess(kernel_fn, noise_var=NOISE_VAR)
        gp.fit(self.X_observed, self.y_observed)
        f_best = float(self.y_observed.min())

        X_cands = np.random.uniform(self.bounds[0], self.bounds[1],
                                     (n_candidates, 1))
        if self.acquisition == "ei":
            acq_vals = expected_improvement(X_cands, gp, f_best, xi=self.xi)
        elif self.acquisition == "ucb":
            acq_vals = upper_confidence_bound(X_cands, gp, kappa=self.kappa)
        elif self.acquisition == "pi":
            acq_vals = probability_of_improvement(X_cands, gp, f_best, xi=self.xi)
        else:
            raise ValueError(f"Unknown acquisition: {self.acquisition}")

        best_cand_idx = int(np.argmax(acq_vals))
        return float(X_cands[best_cand_idx, 0])

    def run(
        self,
        objective: Callable[[float], float],
        kernel_fn: Callable[[np.ndarray, np.ndarray], np.ndarray],
        n_init: int = N_BO_INIT,
        n_iter: int = N_BO_ITER,
    ) -> list[float]:
        '''Run the full Bayesian optimisation loop.

        Args:
            objective: Black-box function to minimise, x (float) -> y (float).
            kernel_fn: Kernel function for the GP surrogate.
            n_init: Number of initial random evaluations.
            n_iter: Number of BO iterations after initialisation.

        Returns:
            List of best-so-far values after each evaluation.
        '''
        rng_run = np.random.default_rng(SEED + 200)
        # Random initialisation
        for _ in range(n_init):
            x_init = float(rng_run.uniform(self.bounds[0], self.bounds[1]))
            y_init = objective(x_init)
            self.observe(x_init, y_init)

        history: list[float] = [self.best_y] * n_init
        for _ in range(n_iter):
            x_next = self.suggest(kernel_fn)
            y_next = objective(x_next)
            self.observe(x_next, y_next)
            history.append(self.best_y)

        return history


print("BayesianOptimizer defined.")
print("  Methods: observe() | suggest() | run()")


### 3.3 Bayesian Optimization Loop — Step-by-Step

We run BO on the Forrester function, visualising the GP surrogate and acquisition
function at each step.  The red star marks the currently suggested next evaluation.


In [ ]:
# ── BO Loop with Step-by-Step Visualisation ────────────────────────────────────
rng_bo = np.random.default_rng(SEED + 300)

# Kernel for BO surrogate
k_bo = lambda X1, X2: rbf_kernel(X1, X2, length_scale=0.15, amplitude=5.0)

# Initialise with 3 random points
n_init_viz = 3
X_bo_init  = rng_bo.uniform(BO_LB, BO_UB, n_init_viz).reshape(-1, 1)
y_bo_init  = forrester(X_bo_init.ravel())

bo_viz = BayesianOptimizer(bounds=(BO_LB, BO_UB), acquisition="ei", xi=0.01)
for xi_b, yi_b in zip(X_bo_init.ravel(), y_bo_init):
    bo_viz.observe(xi_b, yi_b)

x_grid_bo = np.linspace(BO_LB, BO_UB, 400).reshape(-1, 1)
y_true_bo  = forrester(x_grid_bo.ravel())
n_viz_steps = 4

fig_bo, axes_bo = plt.subplots(n_viz_steps, 2, figsize=(14, 4 * n_viz_steps))

for step_b in range(n_viz_steps):
    ax_sur = axes_bo[step_b, 0]  # surrogate
    ax_acq = axes_bo[step_b, 1]  # acquisition

    gp_bo = GaussianProcess(k_bo, noise_var=NOISE_VAR)
    gp_bo.fit(bo_viz.X_observed, bo_viz.y_observed)
    mu_bo, sig_bo = gp_bo.predict(x_grid_bo)

    f_best_bo = float(bo_viz.y_observed.min())
    acq_ei_bo = expected_improvement(x_grid_bo, gp_bo, f_best_bo, xi=0.01)
    x_next_bo = bo_viz.suggest(k_bo)

    # Surrogate plot
    ax_sur.plot(x_grid_bo.ravel(), y_true_bo, "k--", lw=1.5, label="True f(x)")
    ax_sur.plot(x_grid_bo.ravel(), mu_bo, "steelblue", lw=2, label="GP mean")
    ax_sur.fill_between(x_grid_bo.ravel(), mu_bo - 2*sig_bo, mu_bo + 2*sig_bo,
                         alpha=0.3, color="steelblue")
    ax_sur.scatter(bo_viz.X_observed.ravel(), bo_viz.y_observed,
                   c="tomato", s=60, zorder=10, edgecolors="k", lw=0.5)
    ax_sur.axvline(x_next_bo, color="red", ls="--", lw=1.5, label=f"Next: x={x_next_bo:.3f}")
    n_obs_b = len(bo_viz.y_observed)
    ax_sur.set_title(f"Step {step_b + 1}: GP Surrogate ({n_obs_b} obs, best={f_best_bo:.3f})",
                      fontsize=10, fontweight="bold")
    ax_sur.legend(fontsize=8); ax_sur.set_xlim(BO_LB, BO_UB)

    # Acquisition plot
    ax_acq.fill_between(x_grid_bo.ravel(), 0, acq_ei_bo, alpha=0.5, color="darkorange")
    ax_acq.plot(x_grid_bo.ravel(), acq_ei_bo, color="darkorange", lw=2)
    ax_acq.axvline(x_next_bo, color="red", ls="--", lw=1.5)
    ax_acq.scatter([x_next_bo], [float(acq_ei_bo.max())], c="red", s=80, zorder=10)
    ax_acq.set_title(f"Step {step_b + 1}: EI Acquisition", fontsize=10, fontweight="bold")
    ax_acq.set_xlabel("x"); ax_acq.set_ylabel("EI(x)")
    ax_acq.set_xlim(BO_LB, BO_UB)

    # Observe the suggested point
    y_next_bo = float(forrester(x_next_bo))
    bo_viz.observe(x_next_bo, y_next_bo)

plt.suptitle("Bayesian Optimization: Surrogate + Acquisition at Each Step",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"After {n_init_viz + n_viz_steps} evaluations:")
print(f"  Best x found: {bo_viz.best_x:.5f}  (true x*={x_opt_true:.5f})")
print(f"  Best f found: {bo_viz.best_y:.5f}  (true f*={f_opt_true:.5f})")
print(f"  Distance to optimum: |x - x*| = {abs(bo_viz.best_x - x_opt_true):.5f}")


---
## Part 4 — Theory vs Practice: BO vs Baselines

### 4.1 Bayesian Optimization vs Random Search vs Grid Search

**Theoretical prediction:** BO should find the minimum in fewer function evaluations
than random search or grid search, because it uses all past observations to guide
exploration.  The advantage is greatest when evaluations are expensive.

We compare convergence on the Forrester function by running each strategy
$n_{\text{rep}}$ times and computing the **mean best-so-far** across iterations.


In [ ]:
# ── Compare BO vs Random Search vs Grid Search ────────────────────────────────
rng_cmp = np.random.default_rng(SEED + 400)
N_REPS  = 30   # repetitions per strategy
N_TOTAL = N_BO_INIT + N_BO_ITER   # total budget

def run_random_search(n_evals: int, rng_rs: np.random.Generator) -> list[float]:
    '''Run random search and return best-so-far history.

    Args:
        n_evals: Total number of evaluations.
        rng_rs: Random number generator.

    Returns:
        Best-so-far values after each evaluation, length n_evals.
    '''
    xs    = rng_rs.uniform(BO_LB, BO_UB, n_evals)
    ys    = forrester(xs)
    hist  = [float(ys[:i+1].min()) for i in range(n_evals)]
    return hist


def run_grid_search(n_evals: int) -> list[float]:
    '''Run equispaced grid search and return best-so-far history.

    Args:
        n_evals: Number of grid points (evaluations).

    Returns:
        Best-so-far values after each evaluation, length n_evals.
    '''
    xs    = np.linspace(BO_LB, BO_UB, n_evals)
    ys    = forrester(xs)
    hist  = [float(ys[:i+1].min()) for i in range(n_evals)]
    return hist


def run_bo(acquisition_type: str, rng_bo_r: np.random.Generator) -> list[float]:
    '''Run Bayesian optimisation and return best-so-far history.

    Args:
        acquisition_type: "ei", "ucb", or "pi".
        rng_bo_r: Random number generator (used for seeding BO internally).

    Returns:
        Best-so-far values after each evaluation, length N_BO_INIT + N_BO_ITER.
    '''
    bo_r  = BayesianOptimizer(bounds=(BO_LB, BO_UB), acquisition=acquisition_type,
                               xi=0.01, kappa=2.0)
    k_r   = lambda X1, X2: rbf_kernel(X1, X2, 0.15, 5.0)
    hist  = bo_r.run(lambda x: float(forrester(x)), k_r,
                      n_init=N_BO_INIT, n_iter=N_BO_ITER)
    return hist


# Run all strategies across repetitions
histories: dict[str, list[list[float]]] = {
    "BO-EI":    [],
    "BO-UCB":   [],
    "Random":   [],
    "Grid":     [],
}
for rep_i in range(N_REPS):
    rng_rep = np.random.default_rng(SEED + 500 + rep_i)
    np.random.seed(SEED + 500 + rep_i)
    histories["BO-EI"].append(run_bo("ei", rng_rep))
    np.random.seed(SEED + 500 + rep_i)
    histories["BO-UCB"].append(run_bo("ucb", rng_rep))
    histories["Random"].append(run_random_search(N_TOTAL, rng_rep))
    histories["Grid"].append(run_grid_search(N_TOTAL))

# Compute mean and std across reps
mean_hist: dict[str, np.ndarray] = {}
std_hist:  dict[str, np.ndarray] = {}
for method_k, hist_list in histories.items():
    arr = np.array(hist_list)
    mean_hist[method_k] = arr.mean(axis=0)
    std_hist[method_k]  = arr.std(axis=0)

# Visualise convergence
fig_cmp2, ax_cmp2 = plt.subplots(figsize=(10, 6))
cmp_colours = {"BO-EI": "steelblue", "BO-UCB": "darkorange", "Random": "tomato", "Grid": "purple"}
for method_k, col_m in cmp_colours.items():
    iterations = np.arange(1, N_TOTAL + 1)
    ax_cmp2.plot(iterations, mean_hist[method_k], "-", color=col_m, lw=2.5, label=method_k)
    ax_cmp2.fill_between(iterations,
                          mean_hist[method_k] - std_hist[method_k],
                          mean_hist[method_k] + std_hist[method_k],
                          alpha=0.15, color=col_m)

ax_cmp2.axhline(f_opt_true, color="black", ls=":", lw=1.5, label=f"True optimum ({f_opt_true:.2f})")
ax_cmp2.set_xlabel("Number of function evaluations", fontsize=11)
ax_cmp2.set_ylabel("Best f found so far", fontsize=11)
ax_cmp2.set_title(f"BO vs Baselines: Convergence on Forrester Function ({N_REPS} repetitions)",
                   fontsize=11, fontweight="bold")
ax_cmp2.legend(fontsize=9)
ax_cmp2.set_xlim(1, N_TOTAL)
plt.tight_layout()
plt.show()

print(f"Final best-so-far after {N_TOTAL} evaluations (mean over {N_REPS} reps):")
for method_k in ["BO-EI", "BO-UCB", "Random", "Grid"]:
    final_m = mean_hist[method_k][-1]
    gap     = abs(final_m - f_opt_true)
    print(f"  {method_k:>9s}: {final_m:7.4f}  |  gap to optimal: {gap:.4f}")
print(f"  True optimum: {f_opt_true:.4f}")


### 4.2 GP Classification Concepts

For **binary classification**, we want $P(Y=1 \mid \mathbf{x})$.  In GP classification:

1. Place a GP prior on the **latent function** $f(\mathbf{x})$.
2. Pass $f$ through a sigmoid: $P(Y=1 \mid \mathbf{x}) = \sigma(f(\mathbf{x}))$.

The posterior $p(f \mid \mathbf{X}, \mathbf{y})$ is no longer Gaussian (due to the sigmoid
likelihood), requiring approximations such as **Laplace** or **Expectation Propagation**.

As a demonstration, we show that posterior GP samples passed through sigmoid
produce probability estimates with calibrated uncertainty.


In [ ]:
# ── GP Classification: Sigmoid-transformed Posterior Samples ──────────────────
rng_cls = np.random.default_rng(SEED + 600)

# 1D binary classification: positive class when x > 0 (with noise)
X_cls_tr  = rng_cls.uniform(-3, 3, 20).reshape(-1, 1)
y_cls_tr  = (X_cls_tr.ravel() > 0.5).astype(float)   # true boundary at 0.5
y_cls_tr += rng_cls.binomial(1, 0.1, size=20) * (1 - 2 * y_cls_tr)  # 10% label noise
y_cls_tr_signed = 2.0 * y_cls_tr - 1.0   # convert to {-1, +1} for GP latent

# Fit GP on signed labels (approximation: treat as regression)
k_cls    = lambda X1, X2: rbf_kernel(X1, X2, length_scale=1.0, amplitude=2.0)
gp_cls   = GaussianProcess(k_cls, noise_var=0.1)
gp_cls.fit(X_cls_tr, y_cls_tr_signed)

x_cls_test = np.linspace(-4, 4, 200).reshape(-1, 1)
mu_cls, sig_cls = gp_cls.predict(x_cls_test)
posterior_samples_cls = gp_cls.sample_posterior(x_cls_test, n_samples=20)

# Convert latent GP outputs to probabilities via sigmoid
def sigmoid(z: np.ndarray) -> np.ndarray:
    '''Element-wise sigmoid function.

    Args:
        z: Input array, any shape.

    Returns:
        Sigmoid output, same shape as z.
    '''
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))


p_mean_cls    = sigmoid(mu_cls)
p_samples_cls = sigmoid(posterior_samples_cls)
p_lower_cls   = sigmoid(mu_cls - 2 * sig_cls)
p_upper_cls   = sigmoid(mu_cls + 2 * sig_cls)

fig_cls, axes_cls = plt.subplots(1, 2, figsize=(14, 5))

# Left: latent function
axes_cls[0].plot(x_cls_test.ravel(), mu_cls, "steelblue", lw=2, label="GP mean (latent)")
axes_cls[0].fill_between(x_cls_test.ravel(), mu_cls - 2*sig_cls, mu_cls + 2*sig_cls,
                          alpha=0.25, color="steelblue", label="2-sigma band")
for s in range(5):
    axes_cls[0].plot(x_cls_test.ravel(), posterior_samples_cls[:, s],
                     alpha=0.3, lw=1, color="gray")
axes_cls[0].scatter(X_cls_tr.ravel(), y_cls_tr_signed, c=y_cls_tr,
                     cmap="RdBu", s=60, zorder=5, edgecolors="k", lw=0.5)
axes_cls[0].axhline(0, color="gray", ls=":", lw=1)
axes_cls[0].set_title("GP Latent Function for Classification", fontsize=10, fontweight="bold")
axes_cls[0].set_xlabel("x"); axes_cls[0].legend(fontsize=8)

# Right: probability estimates
axes_cls[1].plot(x_cls_test.ravel(), p_mean_cls, "steelblue", lw=2, label="P(Y=1|x) = sigmoid(mu)")
axes_cls[1].fill_between(x_cls_test.ravel(), p_lower_cls, p_upper_cls,
                          alpha=0.3, color="steelblue", label="Sigmoid(mu +/- 2sigma)")
for s in range(10):
    axes_cls[1].plot(x_cls_test.ravel(), p_samples_cls[:, s],
                     alpha=0.2, lw=1, color="gray")
axes_cls[1].scatter(X_cls_tr.ravel(), y_cls_tr, c=y_cls_tr,
                     cmap="RdBu", s=60, zorder=5, edgecolors="k", lw=0.5)
axes_cls[1].axhline(0.5, color="gray", ls=":", lw=1)
axes_cls[1].axvline(0.5, color="gray", ls=":", lw=1, label="True boundary x=0.5")
axes_cls[1].set_title("GP Classification: Probability Estimates", fontsize=10, fontweight="bold")
axes_cls[1].set_xlabel("x"); axes_cls[1].set_ylabel("P(Y=1|x)")
axes_cls[1].legend(fontsize=8)

plt.suptitle("GP Classification via Latent Function + Sigmoid Transformation",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print("The uncertainty band widens near the decision boundary and far from training data.")
print("Full GP classification (Laplace/EP) would marginalize over the latent function.")
# Measure how often the mean prediction is correct
y_pred_cls = (p_mean_cls[np.array([np.argmin(np.abs(x_cls_test.ravel() - xi))
                                    for xi in X_cls_tr.ravel()])] > 0.5).astype(int)
acc_cls = float(np.mean(y_pred_cls == y_cls_tr.astype(int)))
print(f"GP classification accuracy on training data: {acc_cls:.4f}")


### 4.3 BO Convergence via Simple Regret

**Simple regret** $r_t = f(x^*) - f(\hat x_t)$ measures how far the best point found so far is from the true optimum. We run 20 independent trials for EI, UCB, PI, and Random search and plot the mean ± 1 standard deviation of the regret curve on a log scale.


In [ ]:
# ── Bayesian Optimisation Regret Analysis ─────────────────────────────────────
# Simple regret: r_t = f(x*) - f(x_t_best), where x* is the global minimum.
# We compare EI, UCB, PI, and Random search over N_BO_REGRET_RUNS independent
# repetitions and plot mean ± std of the simple regret curve.

N_BO_REGRET_RUNS = 20
forrester_min    = float(np.min(forrester(np.linspace(BO_LB, BO_UB, 10000))))
kernel_regret    = lambda X1, X2: matern52_kernel(X1, X2, length_scale=0.3)

rng_regret = np.random.default_rng(SEED + 50)

regret_curves: dict[str, list[list[float]]] = {
    "EI": [], "UCB": [], "PI": [], "Random": [],
}


def run_single_bo_regret(
    acq_name: str,
    seed_offset: int,
) -> list[float]:
    '''Run one BO trial and return per-iteration simple regret.

    Args:
        acq_name: Acquisition function name ("EI", "UCB", "PI", or "Random").
        seed_offset: RNG seed offset for reproducibility.

    Returns:
        List of simple regret values of length N_BO_INIT + N_BO_ITER.
    '''
    rng_r = np.random.default_rng(SEED + seed_offset)
    x_obs = rng_r.uniform(BO_LB, BO_UB, size=N_BO_INIT)
    y_obs = forrester(x_obs)
    best_y_so_far = float(np.min(y_obs))
    regrets = [best_y_so_far - forrester_min] * N_BO_INIT

    for _ in range(N_BO_ITER):
        if acq_name == "Random":
            x_new = float(rng_r.uniform(BO_LB, BO_UB))
        else:
            acq_lower = acq_name.lower()
            gp_r = GaussianProcess(kernel_regret, noise_var=NOISE_VAR)
            gp_r.fit(x_obs[:, None], y_obs)
            cands = np.linspace(BO_LB, BO_UB, N_BO_CANDIDATES)[:, None]
            if acq_lower == "ei":
                scores = expected_improvement(cands, gp_r, best_y_so_far)
            elif acq_lower == "ucb":
                scores = upper_confidence_bound(cands, gp_r, kappa=2.0)
            else:  # pi
                scores = probability_of_improvement(cands, gp_r, best_y_so_far)
            x_new = float(cands[np.argmin(scores)])
        y_new = float(forrester(np.array([x_new])))
        x_obs = np.append(x_obs, x_new)
        y_obs = np.append(y_obs, y_new)
        best_y_so_far = float(np.min(y_obs))
        regrets.append(best_y_so_far - forrester_min)

    return regrets


for run_idx in range(N_BO_REGRET_RUNS):
    for acq in ["EI", "UCB", "PI", "Random"]:
        regret_curves[acq].append(run_single_bo_regret(acq, seed_offset=run_idx * 10))

# Compute mean and std of regret curves
iterations = list(range(N_BO_INIT + N_BO_ITER))
fig_reg, ax_reg = plt.subplots(figsize=(9, 5))

colors_reg = {"EI": "#1f77b4", "UCB": "#ff7f0e", "PI": "#2ca02c", "Random": "#d62728"}
for acq, color in colors_reg.items():
    arr  = np.array(regret_curves[acq])  # (runs, iters)
    mean = arr.mean(axis=0)
    std  = arr.std(axis=0)
    ax_reg.plot(iterations, mean, label=acq, color=color, lw=2)
    ax_reg.fill_between(iterations, mean - std, mean + std, alpha=0.15, color=color)

ax_reg.axvline(x=N_BO_INIT - 1, color="gray", ls="--", lw=1, label="End of init")
ax_reg.set_xlabel("Iteration", fontsize=11)
ax_reg.set_ylabel("Simple regret  f(x*) - f(best)", fontsize=11)
ax_reg.set_title(f"BO Simple Regret ({N_BO_REGRET_RUNS} runs, Forrester function)",
                  fontsize=12, fontweight="bold")
ax_reg.legend(fontsize=10)
ax_reg.set_yscale("log")
plt.tight_layout()
plt.show()

# Print final regret summary
print("Final simple regret (mean ± std) after last iteration:")
print(f"{'Method':>8s}  {'Mean regret':>12s}  {'Std':>8s}")
print("-" * 34)
for acq in ["EI", "UCB", "PI", "Random"]:
    arr   = np.array(regret_curves[acq])[:, -1]
    print(f"{acq:>8s}  {arr.mean():12.4f}  {arr.std():8.4f}")


---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **The kernel encodes all inductive bias.** Choosing the right kernel (or using
   automatic relevance determination) is the central modelling decision in GP regression.
   LML provides a principled criterion for kernel/hyperparameter selection.

2. **GP posterior is exact for regression.** The closed-form posterior mean and variance
   (via Cholesky) give calibrated uncertainty — 2$\sigma$ bands contain ~95% of
   held-out observations when the model is correctly specified.

3. **Bayesian optimization outperforms random and grid search on expensive functions.**
   By building a GP surrogate and maximising an acquisition function, BO concentrates
   evaluations near the optimum — especially valuable when each evaluation is costly.

4. **EI vs UCB vs PI:** EI is the default (handles scale automatically); UCB with
   $\kappa$ is simpler to tune; PI is greedier and can get stuck.

5. **GP classification requires approximate inference.** The sigmoid likelihood breaks
   the Gaussian conjugacy — Laplace, EP, or MCMC approximations are needed for exact
   posterior. The latent-GP + sigmoid approach shown here is the conceptual foundation.

### What's Next

→ **05-01 (Neural Network Foundations)** begins the deep learning modules.
The GP framework connects forward to neural tangent kernels (NTKs) and neural
processes, which provide a function-space view of neural networks.

### Going Further

- Rasmussen & Williams (2006). *Gaussian Processes for Machine Learning.* MIT Press.
  (The definitive reference; freely available online.)
- Frazier (2018). *A Tutorial on Bayesian Optimization.* arXiv:1807.02811.
- Snoek et al. (2012). *Practical Bayesian Optimization of ML Algorithms.* NeurIPS.
  (Spearmint: GP-BO for neural network hyperparameter tuning.)


In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print("=" * 72)
print("04-10 GAUSSIAN PROCESSES & BAYESIAN OPTIMIZATION — SUMMARY")
print("=" * 72)
print("")
print("1. Kernels:")
print(f"   RBF diag check  : {np.diag(rbf_kernel(x_check, x_check, 1.0, 2.0)).tolist()}")
print(f"   M52 diag check  : {np.diag(matern52_kernel(x_check, x_check, 1.0, 1.5)).tolist()}")
print("")
print("2. GP fit on 1D observations:")
print(f"   LML (l=1.2, a=1.5) = {lml_fit:.4f}")
print(f"   LML (optimal)      = {best_lml:.4f}  at l={best_ls:.4f}, a={best_amp:.4f}")
print("")
print("3. Uncertainty coverage (2-sigma band):")
for mult_r, theory_r, emp_r in coverage_results:
    ok_r = "OK" if abs(emp_r - theory_r) < 0.05 else "WARN"
    print(f"   {mult_r:.1f}s: theory={theory_r:.3f}  empirical={emp_r:.4f}  [{ok_r}]")
print("")
print("4. BO on Forrester: best-so-far after", N_TOTAL, "evaluations")
for method_r in ["BO-EI", "BO-UCB", "Random", "Grid"]:
    print(f"   {method_r:>9s}: {mean_hist[method_r][-1]:.4f}  (optimal={f_opt_true:.4f})")
print("")
print("=" * 72)
assert best_lml > lml_fit, "Optimised LML should exceed initial LML"
assert bo_viz.best_y < 0.0, "BO should find a negative value on Forrester"
assert mean_hist["BO-EI"][-1] <= mean_hist["Random"][-1] + 1.0, "BO-EI should not be much worse than random"
# Coverage checks
for _, theory_r, emp_r in coverage_results:
    assert emp_r >= theory_r - 0.1, f"Coverage {emp_r:.4f} too far below theory {theory_r:.4f}"
print("All assertions passed.")
